# Experiment 4 — Held-out-from-project-adaptation generator (FLUX.1-schnell)

HIT401 Wildlife Deepfake Detection. **RQ4 / Exp 4 question:** does the SDXL-based wildlife adaptation (Exp 3) preserve or improve detection of synthetic wildlife from a different generator **held out from project adaptation**? FLUX.1-schnell is **not** claimed to be unseen by OmniAID (the Flux.1 family is named in OmniAID's Mirage-Train).

Protocol v1.0 (user-approved 2026-09-25): `PROJECT_RECOVERY_STATE/EXPERIMENT_4_PLANNING.md`. All logic lives in versioned scripts in `exp4_heldout_generator/code/`; this notebook only launches them. Order: pilot (21 images, no detector) → main generation (336 predefined seeds) → QC (fixed rules, replacements = next unused seed) → freeze → **fresh runtime** → evaluation of both frozen detectors → analysis.

**Requires** the Colab secret `HF_TOKEN` (Hugging Face read token of an account that accepted the FLUX.1-schnell terms), with notebook access enabled.

In [1]:
from google.colab import drive; drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2; df -h /content | tail -1

Mounted at /content/drive
name, memory.total [MiB]
Tesla T4, 15360 MiB
               total        used        free      shared  buff/cache   available
Mem:              50           2          44           0           3          48
overlay         236G   48G  189G  21% /


In [2]:
# generation dependencies, PINNED to the versions used for the pilot + main generation (see metadata/generation_env_*.json)
!pip install -q diffusers==0.40.0 accelerate==1.15.0 bitsandbytes==0.50.2 transformers==5.16.1 sentencepiece protobuf imagehash
import diffusers, bitsandbytes, transformers, torch; print(torch.__version__, diffusers.__version__, bitsandbytes.__version__, transformers.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 27.5 MB/s eta 0:00:00
2.11.0+cu128 0.40.0 0.50.2 5.16.1


In [6]:
import os, subprocess, sys
from google.colab import userdata
CODE = '/content/drive/MyDrive/HIT401_Wildlife_Deepfake/exp4_heldout_generator/code'
def launch(*args, log='run'):
    # runs a script in the background; the HF token is passed in memory only (never written to disk or printed)
    env = dict(os.environ, HF_TOKEN=userdata.get('HF_TOKEN'), PYTHONUNBUFFERED='1')
    lf = open(f'/content/{log}.log', 'a')
    p = subprocess.Popen([sys.executable, f'{CODE}/' + args[0], *args[1:]], env=env, stdout=lf, stderr=subprocess.STDOUT, cwd=CODE)
    print('started pid', p.pid, '-> /content/' + log + '.log'); return p
def tail(log='run', n=15):
    print(''.join(open(f'/content/{log}.log').readlines()[-n:]))

In [13]:

# REPLACEMENT round: next unused predefined seeds for confirmed QC rejects only (NEEDS_USER never replaced); stop rule 5x quota enforced
p = launch('exp4_generate.py', '--mode', 'replace', '--strategy', 'nf4_fp16', log='replace1')

started pid 43640 -> /content/replace1.log


In [4]:
# Verify access WITHOUT revealing the secret (prints only booleans / public file content)
from huggingface_hub import hf_hub_download
_tok = userdata.get('HF_TOKEN')
print('HF_TOKEN readable by notebook:', _tok is not None and len(_tok) > 20)
_p = hf_hub_download('black-forest-labs/FLUX.1-schnell', 'model_index.json', revision='741f7c3ce8b383c54771c7003378a50191e9efe9', token=_tok)
print('FLUX.1-schnell gated access OK:', '_class_name' in open(_p).read())
del _tok
print('Drive mounted:', os.path.ismount('/content/drive'))
print('Exp 4 scripts:', sorted(os.listdir(CODE)))
print('Exp 4 folders:', sorted(os.listdir(os.path.dirname(CODE))))

HF_TOKEN readable by notebook: True


model_index.json:   0%|          | 0.00/536 [00:00<?, ?B/s]

FLUX.1-schnell gated access OK: True
Drive mounted: True
Exp 4 scripts: ['dataset_v1_prompt_logic.json', 'exp4_analyze.py', 'exp4_common.py', 'exp4_evaluate.py', 'exp4_freeze.py', 'exp4_generate.py', 'exp4_qc.py']
Exp 4 folders: ['candidates', 'code', 'contact_sheets', 'evaluation', 'final', 'metadata', 'pilot']


In [5]:
p = launch('exp4_generate.py', '--mode', 'pilot', '--strategy', 'nf4_fp16', log='pilot')

started pid 2327 -> /content/pilot.log


In [8]:
tail('pilot')


Loading pipeline components...:  29%|██▊       | 2/7 [00:00<00:00,  7.82it/s]

Loading weights: 100%|██████████| 196/196 [00:00<00:00, 2334.99it/s]

Loading pipeline components...: 100%|██████████| 7/7 [00:00<00:00, 12.38it/s]
[2026-09-24T16:05:04+00:00] Pipeline loaded in 150s | transformer + T5 text encoder: bitsandbytes 4-bit NF4 (weights), compute dtype torch.float16 | torch 2.11.0+cu128 diffusers 0.40.0 transformers 5.16.1
[2026-09-24T16:05:27+00:00] pilot flux_opossum_community_690000 22.9s peakVRAM 6.62GiB std 64.2
[2026-09-24T16:05:48+00:00] pilot flux_opossum_camera_day_692000 20.6s peakVRAM 6.62GiB std 54.7
[2026-09-24T16:06:08+00:00] pilot flux_opossum_camera_night_ir_694000 19.9s peakVRAM 6.62GiB std 80.7



In [9]:
# pilot determinism check: regenerate 2 pilot seeds twice, compare pixels (no detector)
p = launch('exp4_generate.py', '--mode', 'determinism', '--strategy', 'nf4_fp16', log='determinism')

started pid 5410 -> /content/determinism.log


In [10]:
# MAIN generation: 336 predefined seeds (n = 0..quota-1 per species x style); resumable; no detector
p = launch('exp4_generate.py', '--mode', 'main', '--strategy', 'nf4_fp16', log='main')

started pid 6716 -> /content/main.log


In [3]:
# ANALYSIS of the locked Exp 4 predictions (no model is run). Primary = fake-only metrics; composite = supplementary.
!cd /content/drive/MyDrive/HIT401_Wildlife_Deepfake/exp4_heldout_generator/code && python3 exp4_analyze.py 2>&1 | tail -80

Traceback (most recent call last):
  File "/content/drive/MyDrive/HIT401_Wildlife_Deepfake/exp4_heldout_generator/code/exp4_analyze.py", line 49, in <module>
    P["paired_outcome"] = np.select([o & a, ~o & ~a, o & ~a, ~o & a],
                          ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                                    ["both_detected", "both_missed", "missed_only_by_adapted", "missed_only_by_original"])
                                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py", line 871, in select
    raise TypeError(msg) from None
TypeError: Choicelist and default value do not have a common dtype: The DType <class 'numpy.dtypes._PyLongDType'> could not be promoted by <class 'numpy.dtypes.StrDType'>. This means that no common DType exists for the given inputs. For example they cannot be stored in a single array unless the dtype is `object`. The